## Generators

### Understanding the execution of Decorator

In [10]:
def deco(func):
    print("Defining decorator")
    def wrapper():
        print("start of wrapper")
        func()
        print("end of wrapper")
    return wrapper

@deco
def my_print():
    print("my print function")

print("function definition ended- function call start")
my_print()
my_print()

Defining decorator
function definition ended- function call start
start of wrapper
my print function
end of wrapper
start of wrapper
my print function
end of wrapper


In the Decorator code above, you can see that the `print("Defining decorator")` is executed only once even though its inside the decorator.

---
**Execution of Decorator:**
1. *Definition Time*: When python reads the def - decorator body runs
2. *Call Time*: each time you call the function , wrapper body runs
---

### Wraps() Func:

When you decorator a function the identity variables such as name points to wrapper, not your original function.
So python reports wrappers details not originals

`wraps(func)` copies those identity cards from original functio instead of wrapper function

### without wraps func

In [4]:
from functools import wraps

def logger(func):
    
    
    def wrapper(*args, **kwargs):
        """Decorator that logs functions"""
        return func(*args, **kwargs)
    return wrapper

@logger
def add(a,b):
    """Adds two numbers"""
    return a+b

print(add.__name__)
print(add.__doc__)
#print(add.__wrapped__)

wrapper
Decorator that logs functions


### with wraps func

In [8]:
from functools import wraps

def logger(func):
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        """Decorator that logs functions"""
        
        return func(*args, **kwargs)
    return wrapper

@logger
def add(a,b):
    """Adds two numbers"""
    return a+b

print(add.__name__)
print(add.__doc__)
print(add.__wrapped__)

add
Adds two numbers
<function add at 0x0000019E1A5789A0>


### Arguments flowing through

In [16]:
def show(func):
    def wrapper(*args, **kwargs):
        print(f"args : {args}, kwargs:{kwargs}")
        return func(*args, **kwargs)
    return wrapper

@show
def greet(name, greeting="Bujji"):
    return f"{greeting}, {name}!"

print(greet("Krishna"))
print(greet("Santosh", greeting= "Madman"))


args : ('Krishna',), kwargs:{}
Bujji, Krishna!
args : ('Santosh',), kwargs:{'greeting': 'Madman'}
Madman, Santosh!


In the above code, you can see that even though there is Default greeting declared as arguement for function, we only get the name passed as output from the decorator.

**Here's why:**

The wrapper only sees the arguments passed at the call site , not the funcion's default values
When `print(greet('Krishna'))` is called, the wrapper gets: *args= ("Krishna", ) and kwargs= {}*.

Function fills in defaults internally

### Modifying return value

In [17]:
def speak(func):
    def wrapper(*args, **kwargs):
        res= func(*args, **kwargs)
        return res.upper() +"!!!"
    return wrapper

@speak
def deliver(msg):
    return msg

print(deliver("Oh my gawdddddd"))
    
    
    

OH MY GAWDDDDDD!!!


### Decorator that blocks call

In [19]:
def only_adults(func):
    print("Age checking started..")
    def wrapper(age):
        if age<18:
            print(f"Not allowed inside, age: {age}")
            return None
        return func(age)
    return wrapper

@only_adults
def enter_pub(age):
    print(f"Allowed inside, age: {age}")

enter_pub(20)
enter_pub(5)
enter_pub(18)
enter_pub(47)

Age checking started..
Allowed inside, age: 20
Not allowed inside, age: 5
Allowed inside, age: 18
Allowed inside, age: 47


### Decorator with arguments

In [22]:
def repeat(times):
    print(f"Decorator set to run {times} times")
    def decorator(func):
        print("Entering decorator")
        def wrapper(*args, **kwargs):
            print(f"Running {func.__name__}")
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(4)
def greet(name):
   print(f"Hello {name}!")

print(greet("Kishi Babu"))


Decorator set to run 4 times
Entering decorator
Running greet
Hello Kishi Babu!
Hello Kishi Babu!
Hello Kishi Babu!
Hello Kishi Babu!
None


### Stacked Decorators

Python applies decorators from the bottom to top execution during definition time and top to bottom execution during the function call


In [ ]:
def A(func):
    print("Inside A")
    def wrapper (*args, **kwargs):
        print("Inside wrapper A")
        res = func(*args, **kwargs)
        print("Exiting wrapper A")
        return res
    print("exiting A")
    return wrapper

def B(func):
    print("Inside B")
    def wrapper (*args, **kwargs):
        print("Inside wrapper B")
        res = func(*args, **kwargs)
        print("Exiting wrapper B")
        return res
    print("Exiting B")
    return wrapper

@A
@B
def greet(name):
    print(f"Hello {name}!")

greet("Kishi")

Inside B
Exiting B
Inside A
exiting A
Inside wrapper A
Inside wrapper B
Hello Kishi!
Exiting wrapper B
Exiting wrapper A


### print vs return

1. `get_five()` returns 5, then the inner and outer decorators both double it: 5×2×2=20

2. `set_five()` prints 5, but the decorator returns the result of print(5), which is None.
3. So None is the final value and the return in wrapper throws *typerror: unsupported operand type(s) for *: 'NoneType' and 'int'*

In [ ]:
def double(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)*2
    return wrapper

@double
@double
def get_five():
    return 5

@double
def set_five():
    print(5)

print(get_five())
set_five()

20
5


TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'

### Example- Decorator


1. Evaluate expressions, top-down	tag("A"),tag("B") then tag("C") setup A, setup B, setup C
2. Apply bottom-up	decorator from C applied, B applied then A	wrap C, wrap B, wrap A
3. Call	A enters, then B, then C, then go, then unwind	enter A, enter B, enter C, go!, exit C, exit B, exit A

`@name` means apply directly; `@name(args)` means call first, then apply the result.

In [32]:
def tag(name):
    print(f"setup {name}")
    def decorator(func):
        print(f"wrap {name}")
        def wrapper():
            print(f"enter {name}")
            func()
            print(f"exit {name}")
        return wrapper
    return decorator

@tag("A")
@tag("B")
@tag("C")
def go():
    print("go!")

go()

setup A
setup B
setup C
wrap C
wrap B
wrap A
enter A
enter B
enter C
go!
exit C
exit B
exit A
